# Polaris Purge Practice — Catalog / Table / View

**Purpose:** Practice drop-with-purge on local OrbStack cluster  
**Observe:** MinIO before/after each purge via http://192.168.139.2:9001  

## What we do
```
Part 1: Setup
  → create practice catalog (purge-enabled)
  → create namespace + table + view

Part 2: Observe MinIO before purge
  → confirm files exist

Part 3: View purge
  → enable DROP_WITH_PURGE on catalog
  → drop view with purgeRequested=true
  → confirm no MinIO change (views have no files)

Part 4: Table purge
  → drop table with purgeRequested=true
  → observe MinIO: Parquet files removed

Part 5: Catalog purge
  → drop catalog with purge
  → observe MinIO: base location files removed

Part 6: Cleanup verification
  → confirm everything gone
```


In [1]:
import sys
sys.path.insert(0, '.')
from polaris_test_utils import *

tok = root_token()
print(f"✅ Root token acquired")
print(f"Polaris: {POLARIS_URL}")
print(f"MinIO:   http://192.168.139.2:9001  (login: minioadmin / minioadmin)")

# Practice resource names (isolated from production + watchdog)
PRACTICE_CATALOG   = "purge-practice"
PRACTICE_NS        = "practice-ns"
PRACTICE_TABLE     = "practice-table"
PRACTICE_VIEW      = "practice-view"
PRACTICE_LOCATION  = "s3a://data-catalog-bucket/purge-practice/"


✅ Polaris test utils loaded
   Polaris: http://192.168.139.2:8181  | Realm: POLARIS
   OpenSearch: https://localhost:9200
✅ Root token acquired
Polaris: http://192.168.139.2:8181
MinIO:   http://192.168.139.2:9001  (login: minioadmin / minioadmin)


## Part 1 — Setup: Create catalog, namespace, table, view

In [2]:
# Step 1: Create practice catalog (purge DISABLED initially — like production)
print("Step 1: Create practice catalog")

r = requests.post(f"{BASE_MGMT}/catalogs", headers=h(tok), json={
    "catalog": {
        "name": PRACTICE_CATALOG,
        "type": "INTERNAL",
        "properties": {
            "default-base-location": PRACTICE_LOCATION,
            "polaris.config.drop-with-purge.enabled": "false",  # disabled initially
        },
        "storageConfigInfo": {
            "storageType": "S3",
            "allowedLocations": [PRACTICE_LOCATION],
            "pathStyleAccess": True,
            "endpoint": "http://192.168.139.2:9000",
            "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
        },
    }
})
print_response(r, "Create catalog")

# Get entityVersion for later PATCH
if r.status_code in (200, 201):
    catalog_version = r.json().get("catalog", {}).get("entityVersion", 1)
    print(f"\n✅ Catalog created (entityVersion: {catalog_version})")


Step 1: Create practice catalog

Create catalog:
  Status:     409
  Request-Id: 626665e0-9248-4452-921e-0535b85a9a56_0000000000000000237
  Body: {
  "error": {
    "message": "Cannot create Catalog purge-practice. Catalog already exists or resolution failed",
    "type": "AlreadyExistsException",
    "code": 409
  }
}


In [3]:
# Step 2: Create namespace
print("Step 2: Create namespace")
r = create_namespace(catalog=PRACTICE_CATALOG, ns=PRACTICE_NS, token=tok)
print_response(r, "Create namespace")


Step 2: Create namespace

Create namespace:
  Status:     409
  Request-Id: 626665e0-9248-4452-921e-0535b85a9a56_0000000000000000238
  Body: {
  "error": {
    "message": "Cannot create namespace practice-ns. Namespace already exists",
    "type": "AlreadyExistsException",
    "code": 409
  }
}


In [4]:
# Step 3: Create table
# This will write Iceberg metadata to MinIO → you'll see it in MinIO UI
print("Step 3: Create table")
r = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables",
    headers=h(tok),
    json={
        "name": PRACTICE_TABLE,
        "location": f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}/",
        "schema": {
            "type": "struct",
            "fields": [
                {"id": 1, "name": "id",      "type": "long",   "required": True},
                {"id": 2, "name": "name",    "type": "string", "required": False},
                {"id": 3, "name": "amount",  "type": "double", "required": False},
            ]
        },
        "properties": {
            "write.format.default": "parquet",
        }
    }
)
print_response(r, "Create table")
if r.status_code in (200, 201):
    metadata_loc = r.json().get("metadata-location", "")
    print(f"\n✅ Table created")
    print(f"   Metadata location: {metadata_loc}")
    print(f"   → Check MinIO: data-catalog-bucket/purge-practice/practice-ns/practice-table/")


Step 3: Create table

Create table:
  Status:     409
  Request-Id: 626665e0-9248-4452-921e-0535b85a9a56_0000000000000000239
  Body: {
  "error": {
    "message": "Table already exists: practice-ns.practice-table",
    "type": "AlreadyExistsException",
    "code": 409
  }
}


In [5]:
# Step 4: Create view
print("Step 4: Create view")
r = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views",
    headers=h(tok),
    json={
        "name": PRACTICE_VIEW,
        "default-namespace": [PRACTICE_NS],
        "schema": {
            "type": "struct",
            "schema-id": 0,
            "fields": [
                {"id": 1, "name": "id",   "type": "long",   "required": True},
                {"id": 2, "name": "name", "type": "string", "required": False},
            ]
        },
        "view-version": {
            "version-id": 1,
            "timestamp-ms": int(__import__("time").time() * 1000),
            "schema-id": 0,
            "default-namespace": [PRACTICE_NS],
            "summary": {"engine": "spark"},
            "representations": [{
                "type": "sql",
                "sql": f"SELECT id, name FROM {PRACTICE_TABLE}",
                "dialect": "spark"
            }]
        },
        "location": f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_VIEW}/",
        "properties": {}
    }
)
print_response(r, "Create view")
if r.status_code in (200, 201):
    print(f"\n✅ View created")
    print(f"   → Views have NO MinIO files (query definition only)")


Step 4: Create view

Create view:
  Status:     409
  Request-Id: 626665e0-9248-4452-921e-0535b85a9a56_0000000000000000240
  Body: {
  "error": {
    "message": "View already exists: practice-ns.practice-view",
    "type": "AlreadyExistsException",
    "code": 409
  }
}


## Part 1b — Write data to table via Iceberg REST commit API

```
Iceberg REST write flow (without Spark):
  1. loadTable   → get current snapshot + storage credentials (STS)
  2. Write Parquet file to MinIO manually (using pyarrow + s3fs)
  3. commitTransaction → register new snapshot referencing the Parquet file

This is what Spark does internally — here we do it step by step.
You will see the Parquet file appear in MinIO after commit.
```


In [6]:
# Install required libraries (run once)
import subprocess
subprocess.run(["pip", "install", "pyarrow", "s3fs", "boto3", "--quiet"], capture_output=True)
print("✅ Libraries ready")


FileNotFoundError: [Errno 2] No such file or directory: 'pip'

In [7]:
# Step A: Load table — get current metadata + STS credentials
print("Step A: Load table (get metadata + STS credentials)")

r_load = requests.get(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
    headers=h(tok)
)
print_response(r_load, "Load table")

table_meta = r_load.json()
metadata_location = table_meta.get("metadata-location", "")
table_location    = table_meta.get("metadata", {}).get("location", f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}")
schema            = table_meta.get("metadata", {}).get("schemas", [{}])[0]
current_snapshot  = table_meta.get("metadata", {}).get("current-snapshot-id", -1)

print(f"\n  metadata-location: {metadata_location}")
print(f"  table-location:    {table_location}")
print(f"  current-snapshot:  {current_snapshot}")


Step A: Load table (get metadata + STS credentials)

Load table:
  Status:     200
  Request-Id: 626665e0-9248-4452-921e-0535b85a9a56_0000000000000000241
  Body: {
  "metadata-location": "s3a://data-catalog-bucket/purge-practice/practice-ns/practice-table/metadata/00001-e9485773-3966-47ea-a905-b1b13a0418f8.metadata.json",
  "metadata": {
    "format-version": 2,
    "table-uuid": "e330d910-a0cf-4671-b53b-381bef50f925",
    "location": "s3a://data-catalog-bucket/purge-practice/practice-ns/practice-table",
    "last-sequence-number": 0,
    "last-updated-ms": 1782200198674,
    "last-column-id": 3,
    "current-schema-id": 0,
    "schemas": [
      {
        "type": "struct",
        "schema-id": 0,
        "fields": [
          {
            "id": 1,
            "name": "id",
            "required": true,
            "type": "long"
          },
          {
            "id": 2,
            "name": "name",
            "required": false,
            "type": "string"
          },
          

In [10]:
# Step B: MinIO credentials (direct — STS not used in company)
print("Step B: MinIO credentials (direct access)")

MINIO_ENDPOINT  = "http://192.168.139.2:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "bhEenDzTEAVPKbukhuY0tRY4WjpZXzh0"

print(f"  endpoint:   {MINIO_ENDPOINT}")
print(f"  access_key: {MINIO_ACCESS_KEY}")
print("  ✅ Ready")

Step B: MinIO credentials (direct access)
  endpoint:   http://192.168.139.2:9000
  access_key: admin
  ✅ Ready


In [11]:
# Step C: Write Parquet file to MinIO using direct credentials
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
import uuid

print("Step C: Write Parquet file to MinIO")

# Sample data matching table schema: id long, name string, amount double
table_data = pa.table({
    "id":     pa.array([1, 2, 3, 4, 5], type=pa.int64()),
    "name":   pa.array(["alice", "bob", "carol", "dave", "eve"], type=pa.string()),
    "amount": pa.array([100.0, 200.5, 300.0, 150.75, 250.0], type=pa.float64()),
})

# File path in MinIO
file_uuid    = uuid.uuid4().hex
data_dir     = f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}/data"
parquet_path = f"{data_dir}/{file_uuid}.parquet"
s3_path      = parquet_path.replace("s3a://", "s3://")

print(f"  Writing to: {s3_path}")

# Connect to MinIO — endpoint_url must go inside client_kwargs
fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    use_ssl=False,
    client_kwargs={
        "endpoint_url": MINIO_ENDPOINT,
    }
)

# Write Parquet
with fs.open(s3_path, "wb") as f_out:
    pq.write_table(table_data, f_out)

FILE_SIZE_BYTES = fs.info(s3_path)["size"]

print(f"  ✅ Parquet written: {FILE_SIZE_BYTES} bytes, {len(table_data)} rows")
print()
print("📸 Check MinIO NOW — .parquet file should be visible:")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")

Step C: Write Parquet file to MinIO
  Writing to: s3://data-catalog-bucket/purge-practice/practice-ns/practice-table/data/1ca3592295a944babde9e2968c690d20.parquet
  ✅ Parquet written: 1097 bytes, 5 rows

📸 Check MinIO NOW — .parquet file should be visible:
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F


In [14]:
# Step D: Write Iceberg manifest + manifest-list to MinIO
# Iceberg v2 requires:
#   1. manifest file (.avro) listing data files
#   2. manifest-list file (.avro) listing manifests
#   3. commit referencing manifest-list path
# We write minimal valid Avro files using fastavro.
import time, struct, uuid, io

print("Step D: Write Iceberg manifest files to MinIO")

try:
    import fastavro
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "fastavro", "--quiet"])
    import fastavro

snapshot_id  = int(time.time() * 1000)
commit_ts    = snapshot_id
seq_num      = 1
table_loc    = f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}"
meta_dir     = f"{table_loc}/metadata"

# ── Manifest file schema (Iceberg v2) ────────────────────────
MANIFEST_SCHEMA = {
    "type": "record",
    "name": "manifest_entry",
    "fields": [
        {"name": "status",          "type": "int"},
        {"name": "snapshot_id",     "type": ["null", "long"],   "default": None},
        {"name": "sequence_number", "type": ["null", "long"],   "default": None},
        {"name": "file_sequence_number", "type": ["null", "long"], "default": None},
        {"name": "data_file", "type": {
            "type": "record", "name": "r2",
            "fields": [
                {"name": "content",             "type": "int"},
                {"name": "file_path",           "type": "string"},
                {"name": "file_format",         "type": "string"},
                {"name": "partition",           "type": {"type": "record", "name": "r102", "fields": []}},
                {"name": "record_count",        "type": "long"},
                {"name": "file_size_in_bytes",  "type": "long"},
                {"name": "column_sizes",        "type": ["null", {"type": "array", "items": {
                    "type": "record", "name": "k81v81",
                    "fields": [{"name": "key", "type": "int"}, {"name": "value", "type": "long"}]
                }}], "default": None},
                {"name": "value_counts",        "type": ["null", {"type": "array", "items": "k81v81"}], "default": None},
                {"name": "null_value_counts",   "type": ["null", {"type": "array", "items": "k81v81"}], "default": None},
                {"name": "nan_value_counts",    "type": ["null", {"type": "array", "items": "k81v81"}], "default": None},
                {"name": "lower_bounds",        "type": ["null", {"type": "array", "items": {
                    "type": "record", "name": "k81v100",
                    "fields": [{"name": "key", "type": "int"}, {"name": "value", "type": "bytes"}]
                }}], "default": None},
                {"name": "upper_bounds",        "type": ["null", {"type": "array", "items": "k81v100"}], "default": None},
                {"name": "key_metadata",        "type": ["null", "bytes"], "default": None},
                {"name": "split_offsets",       "type": ["null", {"type": "array", "items": "long"}], "default": None},
                {"name": "equality_ids",        "type": ["null", {"type": "array", "items": "int"}], "default": None},
                {"name": "sort_order_id",       "type": ["null", "int"], "default": None},
            ]
        }}
    ]
}

# Data file entry for our Parquet file
manifest_entry = {
    "status": 1,  # ADDED
    "snapshot_id": snapshot_id,
    "sequence_number": seq_num,
    "file_sequence_number": seq_num,
    "data_file": {
        "content": 0,  # DATA
        "file_path": parquet_path,
        "file_format": "PARQUET",
        "partition": {},
        "record_count": 5,
        "file_size_in_bytes": FILE_SIZE_BYTES,
        "column_sizes": None,
        "value_counts": None,
        "null_value_counts": None,
        "nan_value_counts": None,
        "lower_bounds": None,
        "upper_bounds": None,
        "key_metadata": None,
        "split_offsets": None,
        "equality_ids": None,
        "sort_order_id": 0,
    }
}

# Write manifest file to MinIO
manifest_path = f"{meta_dir}/{uuid.uuid4().hex}-m0.avro"
s3_manifest   = manifest_path.replace("s3a://", "s3://")

buf = io.BytesIO()
fastavro.writer(buf, fastavro.parse_schema(MANIFEST_SCHEMA), [manifest_entry],
    metadata={
        "schema":              '{"type":"struct","schema-id":0,"fields":[{"id":1,"name":"id","required":true,"type":"long"},{"id":2,"name":"name","required":false,"type":"string"},{"id":3,"name":"amount","required":false,"type":"double"}]}',
        "partition-spec":      '[]',
        "partition-spec-id":   '0',
        "format-version":      '2',
        "content":             'data',
        "sequence-number":     str(seq_num),
        "min-sequence-number": str(seq_num),
        "snapshot-id":         str(snapshot_id),
        "added-data-files":    '1',
        "existing-data-files": '0',
        "deleted-data-files":  '0',
        "added-rows":          '5',
        "existing-rows":       '0',
        "deleted-rows":        '0',
    }
)
manifest_bytes = buf.getvalue()
manifest_size  = len(manifest_bytes)

with fs.open(s3_manifest, "wb") as f_out:
    f_out.write(manifest_bytes)
print(f"  ✅ manifest written: {s3_manifest} ({manifest_size} bytes)")

# ── Manifest-list schema ──────────────────────────────────────
MANIFEST_LIST_SCHEMA = {
    "type": "record",
    "name": "manifest_file",
    "fields": [
        {"name": "manifest_path",         "type": "string"},
        {"name": "manifest_length",        "type": "long"},
        {"name": "partition_spec_id",      "type": "int"},
        {"name": "content",                "type": "int",  "default": 0},
        {"name": "sequence_number",        "type": "long", "default": 0},
        {"name": "min_sequence_number",    "type": "long", "default": 0},
        {"name": "added_snapshot_id",      "type": "long"},
        {"name": "added_data_files_count", "type": "int"},
        {"name": "existing_data_files_count","type": "int"},
        {"name": "deleted_data_files_count","type": "int"},
        {"name": "added_rows_count",       "type": "long"},
        {"name": "existing_rows_count",    "type": "long"},
        {"name": "deleted_rows_count",     "type": "long"},
        {"name": "partitions",             "type": ["null", {"type": "array", "items": {
            "type": "record", "name": "r508",
            "fields": [
                {"name": "contains_null", "type": "boolean"},
                {"name": "contains_nan",  "type": ["null", "boolean"], "default": None},
                {"name": "lower_bound",   "type": ["null", "bytes"],   "default": None},
                {"name": "upper_bound",   "type": ["null", "bytes"],   "default": None},
            ]
        }}], "default": None},
        {"name": "key_metadata", "type": ["null", "bytes"], "default": None},
    ]
}

manifest_list_entry = {
    "manifest_path":              manifest_path,
    "manifest_length":            manifest_size,
    "partition_spec_id":          0,
    "content":                    0,
    "sequence_number":            seq_num,
    "min_sequence_number":        seq_num,
    "added_snapshot_id":          snapshot_id,
    "added_data_files_count":     1,
    "existing_data_files_count":  0,
    "deleted_data_files_count":   0,
    "added_rows_count":           5,
    "existing_rows_count":        0,
    "deleted_rows_count":         0,
    "partitions":                 None,
    "key_metadata":               None,
}

# Write manifest-list file
manifest_list_path = f"{meta_dir}/snap-{snapshot_id}-{uuid.uuid4().hex}.avro"
s3_manifest_list   = manifest_list_path.replace("s3a://", "s3://")

buf2 = io.BytesIO()
fastavro.writer(buf2, fastavro.parse_schema(MANIFEST_LIST_SCHEMA), [manifest_list_entry])
with fs.open(s3_manifest_list, "wb") as f_out:
    f_out.write(buf2.getvalue())
print(f"  ✅ manifest-list written: {s3_manifest_list}")
print()
print("📸 Check MinIO — metadata/ folder should have .avro files now")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")

Step D: Write Iceberg manifest files to MinIO
  ✅ manifest written: s3://data-catalog-bucket/purge-practice/practice-ns/practice-table/metadata/4650e9977e704272b57a0984074a2266-m0.avro (2506 bytes)
  ✅ manifest-list written: s3://data-catalog-bucket/purge-practice/practice-ns/practice-table/metadata/snap-1782200451901-e86f9ca7f6dd446eb7a2d00e68ca4deb.avro

📸 Check MinIO — metadata/ folder should have .avro files now
   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F


In [15]:
# Step D2: Commit snapshot referencing manifest-list
print("Step D2: Commit snapshot")

r_commit = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
    headers=h(tok),
    json={
        "identifier": {
            "namespace": [PRACTICE_NS],
            "name": PRACTICE_TABLE
        },
        "requirements": [
            {
                "type": "assert-current-snapshot-id",
                "snapshot-id": current_snapshot
            }
        ],
        "updates": [
            {
                "action": "add-snapshot",
                "snapshot": {
                    "snapshot-id":     snapshot_id,
                    "sequence-number": seq_num,
                    "timestamp-ms":    commit_ts,
                    "summary": {
                        "operation":              "append",
                        "added-data-files":       "1",
                        "added-records":          "5",
                        "added-files-size":       str(FILE_SIZE_BYTES),
                        "total-data-files":       "1",
                        "total-records":          "5",
                        "total-files-size":       str(FILE_SIZE_BYTES),
                        "total-delete-files":     "0",
                        "total-equality-deletes": "0",
                        "total-position-deletes": "0",
                    },
                    "manifest-list": manifest_list_path,
                    "schema-id": 0,
                }
            },
            {
                "action": "set-snapshot-ref",
                "ref-name": "main",
                "type": "branch",
                "snapshot-id": snapshot_id
            }
        ]
    }
)
print_response(r_commit, "Commit snapshot")
if r_commit.status_code in (200, 201, 204):
    print(f"\n✅ Snapshot {snapshot_id} committed — 5 rows visible")
    print(f"   manifest-list: {manifest_list_path}")

Step D2: Commit snapshot

Commit snapshot:
  Status:     400
  Request-Id: 626665e0-9248-4452-921e-0535b85a9a56_0000000000000000242
  Body: {
  "error": {
    "message": "HTTP 400 Bad Request",
    "type": "WebApplicationException",
    "code": 400
  }
}


In [ ]:
# Step E: Verify — load table again and check new snapshot
print("Step E: Verify new snapshot registered")

r_verify = requests.get(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
    headers=h(tok)
)
meta = r_verify.json().get("metadata", {})
new_snapshot_id = meta.get("current-snapshot-id", -1)
snapshots       = meta.get("snapshots", [])

print(f"  current-snapshot-id: {new_snapshot_id}")
print(f"  total snapshots:     {len(snapshots)}")
print(f"  snapshot changed:    {'✅' if new_snapshot_id != current_snapshot else '❌ (same as before)'}")

for s in snapshots:
    summary = s.get("summary", {})
    print(f"  snapshot {s.get('snapshot-id')}: "
          f"op={summary.get('operation')} "
          f"records={summary.get('added-records', '?')}")


In [17]:
# Check request ID in Polaris logs
REQUEST_ID = "626665e0-9248-4452-921e-0535b85a9a56_0000000000000000242"

result = client.search(
    index="kube-fb-log",
    body={
        "query": {
            "bool": {
                "must": [
                    {"range": {"@timestamp": {"gte": "now-5m"}}},
                    {"term": {"kubernetes.container_name.keyword": "benchmarks-polaris"}},
                    {"match": {"message": REQUEST_ID[:36]}},  # match on UUID prefix
                ]
            }
        },
        "size": 5,
    }
)
print_logs(result["hits"]["hits"])

SSLError: ConnectionError([SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)) caused by: SSLError([SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010))

In [21]:
# Variant 3: check what your existing test notebooks use for commit
# Look at how polaris_test_utils cleanup() deletes tables — may hint at write path
import inspect
print(inspect.getsource(cleanup))

def cleanup(token=None):
    """
    Delete ALL test resources following dependency order:
      1. Tables + Views   (inside namespaces)
      2. Namespaces       (inside catalogs)
      3. Catalog Roles    (inside catalogs, skip catalog_admin)
      4. Catalogs
      5. Principals       (skip root)
      6. Principal Roles  (skip service_admin)

    Mirrors polaris_clean_all.ipynb logic.
    Safe: skips system entities, ignores 404s.
    """
    tok = token or root_token()
    print("🧹 Cleaning up all test resources...")

    # ── Fetch current state ───────────────────────────────────
    catalogs = requests.get(
        f"{BASE_MGMT}/catalogs", headers=h(tok)
    ).json().get("catalogs", [])

    principals = requests.get(
        f"{BASE_MGMT}/principals", headers=h(tok)
    ).json().get("principals", [])

    principal_roles = requests.get(
        f"{BASE_MGMT}/principal-roles", headers=h(tok)
    ).json().get("roles", [])

    # ── Step 1+2: Tables + Views + Namespaces (recursiv

## Part 2 — Observe MinIO before purge

```
Open MinIO UI: http://192.168.139.2:9001
Navigate to: data-catalog-bucket/purge-practice/

You should see:
  practice-ns/
    practice-table/
      metadata/    ← Iceberg manifest files (.json, .avro)
      data/        ← Parquet files (empty if no data written)
    practice-view/ ← view metadata only (no Parquet)
```


In [ ]:
# Verify table + view exist via Polaris API
print("Verifying entities exist...")

r_t = requests.get(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
    headers=h(tok)
)
r_v = requests.get(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views/{PRACTICE_VIEW}",
    headers=h(tok)
)
print(f"Table {PRACTICE_TABLE}: HTTP {r_t.status_code} {'✅' if r_t.status_code == 200 else '❌'}")
print(f"View  {PRACTICE_VIEW}:  HTTP {r_v.status_code} {'✅' if r_v.status_code == 200 else '❌'}")
print()
print("📸 Check MinIO NOW (before purge)")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")


## Part 3 — View purge (no MinIO effect)

In [ ]:
# Try DROP VIEW without purge flag first (should work on any catalog)
print("Step 3a: Drop view WITHOUT purge (no flag)")
r = requests.delete(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views/{PRACTICE_VIEW}",
    headers=h(tok)
)
print_response(r, "Drop view (no purge)")

# Recreate for purge test
print("\nRecreating view for purge test...")
r2 = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views",
    headers=h(tok),
    json={
        "name": PRACTICE_VIEW,
        "default-namespace": [PRACTICE_NS],
        "schema": {
            "type": "struct",
            "schema-id": 0,
            "fields": [{"id": 1, "name": "id", "type": "long", "required": True}]
        },
        "view-version": {
            "version-id": 1,
            "timestamp-ms": int(__import__("time").time() * 1000),
            "schema-id": 0,
            "default-namespace": [PRACTICE_NS],
            "summary": {"engine": "spark"},
            "representations": [{"type": "sql", "sql": f"SELECT id FROM {PRACTICE_TABLE}", "dialect": "spark"}]
        },
        "location": f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_VIEW}/",
        "properties": {}
    }
)
print_response(r2, "Recreate view")


In [ ]:
# Enable DROP_WITH_PURGE on catalog (needed for purgeRequested=true)
print("Step 3b: Enable DROP_WITH_PURGE on catalog")

# Get current entityVersion
r_get = requests.get(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok))
catalog_version = r_get.json().get("catalog", {}).get("entityVersion", 1)
print(f"Current entityVersion: {catalog_version}")

r_patch = requests.patch(
    f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}",
    headers=h(tok),
    json={
        "currentEntityVersion": catalog_version,
        "properties": {
            "polaris.config.drop-with-purge.enabled": "true"
        }
    }
)
print_response(r_patch, "Enable purge")


In [ ]:
# Drop VIEW with purge
print("Step 3c: Drop view WITH purgeRequested=true")
r = requests.delete(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views/{PRACTICE_VIEW}?purgeRequested=true",
    headers=h(tok)
)
print_response(r, "Drop view (purge)")
print()
print("📸 Check MinIO AFTER view purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print()
print("Expected: NO change in MinIO")
print("   → views have no Parquet files")
print("   → purge only removes view metadata from PostgreSQL")


## Part 4 — Table purge (MinIO files removed)

In [ ]:
# Drop TABLE with purge
print("Step 4: Drop table WITH purgeRequested=true")
print()
print("📸 Check MinIO BEFORE (take note of files present)")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print()

r = requests.delete(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}?purgeRequested=true",
    headers=h(tok)
)
print_response(r, "Drop table (purge)")
print()
print("📸 Check MinIO AFTER table purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print()
print("Expected: Iceberg metadata files GONE")
print("   metadata/*.json → deleted")
print("   metadata/*.avro → deleted")
print("   data/*.parquet  → deleted (if any data was written)")


In [ ]:
# Verify table gone from Polaris
r = requests.get(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
    headers=h(tok)
)
print(f"Table after purge: HTTP {r.status_code} {'✅ (404 = gone)' if r.status_code == 404 else '❌ (still exists)'}")

# Namespace still exists
r2 = requests.get(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}",
    headers=h(tok)
)
print(f"Namespace after table purge: HTTP {r2.status_code} {'✅ (still exists)' if r2.status_code == 200 else '?'}")
print()
print("Note: dropping table does NOT drop namespace")
print("      must drop bottom-up: table → namespace → catalog")


## Part 5 — Catalog purge (base location removed from MinIO)

In [ ]:
# Must drop namespace before catalog
print("Step 5a: Drop namespace (required before catalog drop)")
r = requests.delete(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}",
    headers=h(tok)
)
print_response(r, "Drop namespace")


In [ ]:
# Drop CATALOG with purge
# Note: catalog drop purge removes the base location from MinIO
print("Step 5b: Drop catalog WITH purgeRequested=true")
print()
print("📸 Check MinIO BEFORE catalog purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print()

r = requests.delete(
    f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}?purgeRequested=true",
    headers=h(tok)
)
print_response(r, "Drop catalog (purge)")
print()
print("📸 Check MinIO AFTER catalog purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print()
print("Expected: purge-practice/ prefix GONE from MinIO")


## Part 6 — Verification

In [ ]:
# Verify everything gone
print("Final verification")
print()

# Catalog
r = requests.get(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok))
print(f"Catalog {PRACTICE_CATALOG}: HTTP {r.status_code} {'✅ (404 = gone)' if r.status_code == 404 else '❌ (still exists)'}")

# List all catalogs (confirm not in list)
r2 = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok))
all_cats = [c["name"] for c in r2.json().get("catalogs", [])]
print(f"All catalogs: {all_cats}")
print(f"purge-practice in list: {'❌ (should be gone)' if PRACTICE_CATALOG in all_cats else '✅ (removed)'}")


## Summary — What we learned

```
View  drop + purge:
  → Polaris: view definition removed from PostgreSQL ✅
  → MinIO:   NO change (views have no files)
  → Purge flag for views = safety gate only, not storage deletion

Table drop + purge:
  → Polaris: table metadata removed from PostgreSQL ✅
  → MinIO:   Iceberg metadata + Parquet files DELETED ✅
  → Irreversible — data cannot be recovered after purge

Catalog drop + purge:
  → Polaris: all catalog metadata removed from PostgreSQL ✅
  → MinIO:   base location prefix files DELETED ✅
  → Must drop all tables + namespaces first
  → Irreversible

Order (bottom-up, always):
  table → namespace → catalog

DROP_WITH_PURGE on production catalogs:
  → DISABLE (false) — protect real data from accidental purge
  → only enable for test/watchdog catalogs
```
